In [1]:
import requests
import pandas as pd
import sqlite3

# -----------------------------------
# STEP 1: Fetch Bitcoin Data from API
# -----------------------------------

url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

params = {
    "vs_currency": "usd",
    "days": "365"
}

response = requests.get(url, params=params)

# Convert response to JSON
data = response.json()

# Extract price data
prices = data["prices"]

# -----------------------------------
# STEP 2: Create DataFrame
# -----------------------------------

df = pd.DataFrame(prices, columns=["timestamp", "price"])

# Convert timestamp to readable date
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

# Display sample data
print("Sample Data:")
print(df.head())

# -----------------------------------
# STEP 3: Store Data in SQLite Database
# -----------------------------------

# Create/connect database
conn = sqlite3.connect("bitcoin_prices.db")

# Save to SQL table
df.to_sql(
    name="bitcoin_prices",
    con=conn,
    if_exists="replace",
    index=False
)

print("\nData stored in SQLite database successfully!")

# -----------------------------------
# STEP 4: Read Data from Database
# -----------------------------------

query = """
SELECT *
FROM bitcoin_prices;
"""

sql_df = pd.read_sql(query, conn)

print("\nData fetched from database:")
print(sql_df.head())

# -----------------------------------
# STEP 5: Convert Database Data to CSV
# -----------------------------------

csv_file_name = "bitcoin_prices.csv"

sql_df.to_csv(csv_file_name, index=False)

print(f"\nCSV file saved successfully as: {csv_file_name}")

# -----------------------------------
# STEP 6: Close Database Connection
# -----------------------------------

conn.close()

print("\nProcess Completed Successfully!")

Sample Data:
   timestamp          price
0 2025-05-13  102876.830429
1 2025-05-14  104184.490393
2 2025-05-15  103594.425751
3 2025-05-16  103708.851364
4 2025-05-17  103556.034940

Data stored in SQLite database successfully!

Data fetched from database:
             timestamp          price
0  2025-05-13 00:00:00  102876.830429
1  2025-05-14 00:00:00  104184.490393
2  2025-05-15 00:00:00  103594.425751
3  2025-05-16 00:00:00  103708.851364
4  2025-05-17 00:00:00  103556.034940

CSV file saved successfully as: bitcoin_prices.csv

Process Completed Successfully!


Bitcoin data stored successfully!
Monitoring table updated successfully!

Last Collection Records:
   id collection_date collection_timestamp  total_records
0   1      2026-05-12  2026-05-12 09:37:03            366

Process Completed Successfully!


In [5]:
import requests
import pandas as pd
import sqlite3
from datetime import datetime

# -----------------------------------
# STEP 1: Setup Date and File Names
# -----------------------------------

current_date = datetime.now().strftime("%Y%m%d")
current_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Database and CSV names
db_name = f"bitcoin_prices_{current_date}.db"
csv_file_name = f"bitcoin_prices_{current_date}.csv"

# -----------------------------------
# STEP 2: Fetch Bitcoin Data
# -----------------------------------

url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

params = {
    "vs_currency": "usd",
    "days": "365"
}

response = requests.get(url, params=params)

# Convert JSON response
data = response.json()

# Extract prices
prices = data["prices"]

# -----------------------------------
# STEP 3: Create DataFrame
# -----------------------------------

df = pd.DataFrame(prices, columns=["timestamp", "price"])

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

print("Sample Data:")
print(df.head())

# -----------------------------------
# STEP 4: Connect SQLite Database
# -----------------------------------

conn = sqlite3.connect(db_name)

cursor = conn.cursor()

# -----------------------------------
# STEP 5: Store Bitcoin Data
# -----------------------------------

df.to_sql(
    name="bitcoin_prices",
    con=conn,
    if_exists="replace",
    index=False
)

print(f"\nDatabase Created: {db_name}")

# -----------------------------------
# STEP 6: Create Monitoring Table
# -----------------------------------

cursor.execute("""
CREATE TABLE IF NOT EXISTS data_collection_monitor (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    collection_date TEXT,
    collection_timestamp TEXT,
    total_records INTEGER,
    database_name TEXT,
    csv_file_name TEXT
)
""")

# -----------------------------------
# STEP 7: Insert Monitoring Data
# -----------------------------------

total_records = len(df)

cursor.execute("""
INSERT INTO data_collection_monitor (
    collection_date,
    collection_timestamp,
    total_records,
    database_name,
    csv_file_name
)
VALUES (?, ?, ?, ?, ?)
""", (
    current_date,
    current_timestamp,
    total_records,
    db_name,
    csv_file_name
))

# Save changes
conn.commit()

print("\nMonitoring table updated successfully!")

# -----------------------------------
# STEP 8: Read Bitcoin Data
# -----------------------------------

query = """
SELECT *
FROM bitcoin_prices;
"""

sql_df = pd.read_sql(query, conn)

print("\nData from Database:")
print(sql_df.head())

# -----------------------------------
# STEP 9: Save as CSV
# -----------------------------------

sql_df.to_csv(csv_file_name, index=False)

print(f"\nCSV File Saved: {csv_file_name}")

# -----------------------------------
# STEP 10: View Monitoring Table
# -----------------------------------

monitor_query = """
SELECT *
FROM data_collection_monitor
ORDER BY id DESC;
"""

monitor_df = pd.read_sql(monitor_query, conn)

print("\nMonitoring Table:")
print(monitor_df)

# -----------------------------------
# STEP 11: Close Connection
# -----------------------------------

conn.close()

print("\nProcess Completed Successfully!")

Sample Data:
   timestamp          price
0 2025-05-13  102876.830429
1 2025-05-14  104184.490393
2 2025-05-15  103594.425751
3 2025-05-16  103708.851364
4 2025-05-17  103556.034940

Database Created: bitcoin_prices_20260512.db

Monitoring table updated successfully!

Data from Database:
             timestamp          price
0  2025-05-13 00:00:00  102876.830429
1  2025-05-14 00:00:00  104184.490393
2  2025-05-15 00:00:00  103594.425751
3  2025-05-16 00:00:00  103708.851364
4  2025-05-17 00:00:00  103556.034940

CSV File Saved: bitcoin_prices_20260512.csv

Monitoring Table:
   id collection_date collection_timestamp  total_records  \
0   1        20260512  2026-05-12 09:38:58            366   

                database_name                csv_file_name  
0  bitcoin_prices_20260512.db  bitcoin_prices_20260512.csv  

Process Completed Successfully!


In [7]:
import requests
import pandas as pd
import sqlite3
from datetime import datetime

# Monitoring DB
monitor_conn = sqlite3.connect("monitoring.db")

# Create monitoring table
monitor_conn.execute("""
CREATE TABLE IF NOT EXISTS monitor (
    collection_date TEXT
)
""")

# Get last collection date
last_date = monitor_conn.execute(
    "SELECT collection_date FROM monitor ORDER BY ROWID DESC LIMIT 1"
).fetchone()

today = datetime.now()

create_new_db = False

# Check 10-day condition
if last_date is None:
    create_new_db = True
else:
    last_collection = datetime.strptime(last_date[0], "%Y-%m-%d")
    days_diff = (today - last_collection).days

    if days_diff >= 10:
        create_new_db = True

# Create new DB if needed
if create_new_db:

    current_date = today.strftime("%Y%m%d")

    db_name = f"bitcoin_{current_date}.db"
    csv_name = f"bitcoin_{current_date}.csv"

    # Fetch Bitcoin data
    url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

    params = {
        "vs_currency": "usd",
        "days": "365"
    }

    response = requests.get(url, params=params)

    data = response.json()

    prices = data["prices"]

    # Create DataFrame
    df = pd.DataFrame(prices, columns=["timestamp", "price"])

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    # Create Bitcoin DB
    conn = sqlite3.connect(db_name)

    df.to_sql(
        "bitcoin_prices",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

    # Convert to CSV
    df.to_csv(csv_name, index=False)

    # Update monitoring table
    monitor_conn.execute(
        "INSERT INTO monitor VALUES (?)",
        (today.strftime("%Y-%m-%d"),)
    )

    monitor_conn.commit()

    print(f"New DB Created: {db_name}")
    print(f"CSV Created: {csv_name}")

else:
    print("Less than 10 days. No new DB created.")

# Close monitoring DB
monitor_conn.close()

New DB Created: bitcoin_20260512.db
CSV Created: bitcoin_20260512.csv


In [1]:
import requests
import pandas as pd
import sqlite3
from datetime import datetime
from statsmodels.tsa.arima.model import ARIMA

# -----------------------------------
# STEP 1: Monitoring DB
# -----------------------------------

monitor_conn = sqlite3.connect("monitoring.db")

monitor_conn.execute("""
CREATE TABLE IF NOT EXISTS monitor (
    collection_date TEXT
)
""")

last_date = monitor_conn.execute(
    "SELECT collection_date FROM monitor ORDER BY ROWID DESC LIMIT 1"
).fetchone()

today = datetime.now()

create_new_db = False

# -----------------------------------
# STEP 2: Check 10-Day Condition
# -----------------------------------

if last_date is None:
    create_new_db = True
else:
    last_collection = datetime.strptime(last_date[0], "%Y-%m-%d")
    days_diff = (today - last_collection).days

    if days_diff >= 10:
        create_new_db = True

# -----------------------------------
# STEP 3: Create New Bitcoin DB
# -----------------------------------

if create_new_db:

    current_date = today.strftime("%Y%m%d")

    bitcoin_db = f"bitcoin_{current_date}.db"
    bitcoin_csv = f"bitcoin_{current_date}.csv"

    # Fetch Bitcoin Data
    url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

    params = {
        "vs_currency": "usd",
        "days": "365"
    }

    response = requests.get(url, params=params)

    data = response.json()

    prices = data["prices"]

    # Create DataFrame
    df = pd.DataFrame(prices, columns=["timestamp", "price"])

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    # Save Bitcoin DB
    conn = sqlite3.connect(bitcoin_db)

    df.to_sql(
        "bitcoin_prices",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

    # Save Bitcoin CSV
    df.to_csv(bitcoin_csv, index=False)

    # Update monitoring DB
    monitor_conn.execute(
        "INSERT INTO monitor VALUES (?)",
        (today.strftime("%Y-%m-%d"),)
    )

    monitor_conn.commit()

    print(f"Bitcoin DB Created: {bitcoin_db}")
    print(f"Bitcoin CSV Created: {bitcoin_csv}")

else:
    print("Less than 10 days. Using existing data.")

# -----------------------------------
# STEP 4: Load CSV for ARIMA
# -----------------------------------

df = pd.read_csv(bitcoin_csv)

df["timestamp"] = pd.to_datetime(df["timestamp"])

df.set_index("timestamp", inplace=True)

series = df["price"]

# -----------------------------------
# STEP 5: Train ARIMA Model
# -----------------------------------

model = ARIMA(series, order=(5,1,0))

model_fit = model.fit()

# Forecast next 10 days
forecast = model_fit.forecast(steps=10)

# Future Dates
future_dates = pd.date_range(
    start=series.index[-1] + pd.Timedelta(days=1),
    periods=10
)

# Forecast DataFrame
forecast_df = pd.DataFrame({
    "date": future_dates,
    "predicted_price": forecast
})

print("\n10-Day Forecast:")
print(forecast_df)

# -----------------------------------
# STEP 6: Create Prediction DB
# -----------------------------------

prediction_db = f"bitcoin_prediction_{current_date}.db"

prediction_conn = sqlite3.connect(prediction_db)

forecast_df.to_sql(
    "bitcoin_predictions",
    prediction_conn,
    if_exists="replace",
    index=False
)

prediction_conn.close()

print(f"\nPrediction DB Created: {prediction_db}")

# -----------------------------------
# STEP 7: Save Forecast CSV
# -----------------------------------

forecast_csv = f"bitcoin_forecast_{current_date}.csv"

forecast_df.to_csv(forecast_csv, index=False)

print(f"Forecast CSV Created: {forecast_csv}")

# -----------------------------------
# STEP 8: Close Monitoring DB
# -----------------------------------

monitor_conn.close()

print("\nProcess Completed Successfully!")

Bitcoin DB Created: bitcoin_20260512.db
Bitcoin CSV Created: bitcoin_20260512.csv

10-Day Forecast:
                   date  predicted_price
366 2026-05-13 04:25:34     81134.499238
367 2026-05-14 04:25:34     81141.663526
368 2026-05-15 04:25:34     81150.677081
369 2026-05-16 04:25:34     81150.828081
370 2026-05-17 04:25:34     81148.519132
371 2026-05-18 04:25:34     81148.408066
372 2026-05-19 04:25:34     81148.439098
373 2026-05-20 04:25:34     81148.482680
374 2026-05-21 04:25:34     81148.490418
375 2026-05-22 04:25:34     81148.481483


c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is ava


Prediction DB Created: bitcoin_prediction_20260512.db
Forecast CSV Created: bitcoin_forecast_20260512.csv

Process Completed Successfully!


In [2]:
import requests
import pandas as pd
import sqlite3
from datetime import datetime
from statsmodels.tsa.arima.model import ARIMA

# -----------------------------------
# STEP 1: Monitoring DB
# -----------------------------------

monitor_conn = sqlite3.connect("monitoring.db")

# Table for latest collection check
monitor_conn.execute("""
CREATE TABLE IF NOT EXISTS monitor (
    collection_date TEXT
)
""")

# New table to store ALL insertion history
monitor_conn.execute("""
CREATE TABLE IF NOT EXISTS insertion_history (
    insertion_date TEXT,
    bitcoin_db TEXT,
    prediction_db TEXT,
    bitcoin_csv TEXT,
    forecast_csv TEXT
)
""")

# -----------------------------------
# STEP 2: Get Last Collection Date
# -----------------------------------

last_date = monitor_conn.execute(
    "SELECT collection_date FROM monitor ORDER BY ROWID DESC LIMIT 1"
).fetchone()

today = datetime.now()

create_new_db = False

# -----------------------------------
# STEP 3: Check 10-Day Condition
# -----------------------------------

if last_date is None:
    create_new_db = True
else:
    last_collection = datetime.strptime(last_date[0], "%Y-%m-%d")
    days_diff = (today - last_collection).days

    if days_diff >= 10:
        create_new_db = True

# -----------------------------------
# STEP 4: Create New Data
# -----------------------------------

if create_new_db:

    current_date = today.strftime("%Y%m%d")

    bitcoin_db = f"bitcoin_{current_date}.db"

    prediction_db = f"bitcoin_prediction_{current_date}.db"

    bitcoin_csv = f"bitcoin_{current_date}.csv"

    forecast_csv = f"bitcoin_forecast_{current_date}.csv"

    # -----------------------------------
    # Fetch Bitcoin Data
    # -----------------------------------

    url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

    params = {
        "vs_currency": "usd",
        "days": "365"
    }

    response = requests.get(url, params=params)

    data = response.json()

    prices = data["prices"]

    # -----------------------------------
    # Create DataFrame
    # -----------------------------------

    df = pd.DataFrame(prices, columns=["timestamp", "price"])

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    # -----------------------------------
    # Save Bitcoin DB
    # -----------------------------------

    conn = sqlite3.connect(bitcoin_db)

    df.to_sql(
        "bitcoin_prices",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

    # -----------------------------------
    # Save Bitcoin CSV
    # -----------------------------------

    df.to_csv(bitcoin_csv, index=False)

    # -----------------------------------
    # ARIMA Forecasting
    # -----------------------------------

    df.set_index("timestamp", inplace=True)

    series = df["price"]

    model = ARIMA(series, order=(5,1,0))

    model_fit = model.fit()

    forecast = model_fit.forecast(steps=10)

    future_dates = pd.date_range(
        start=series.index[-1] + pd.Timedelta(days=1),
        periods=10
    )

    forecast_df = pd.DataFrame({
        "date": future_dates,
        "predicted_price": forecast
    })

    # -----------------------------------
    # Save Prediction DB
    # -----------------------------------

    prediction_conn = sqlite3.connect(prediction_db)

    forecast_df.to_sql(
        "bitcoin_predictions",
        prediction_conn,
        if_exists="replace",
        index=False
    )

    prediction_conn.close()

    # -----------------------------------
    # Save Forecast CSV
    # -----------------------------------

    forecast_df.to_csv(forecast_csv, index=False)

    # -----------------------------------
    # Update Monitor Table
    # -----------------------------------

    monitor_conn.execute(
        "INSERT INTO monitor VALUES (?)",
        (today.strftime("%Y-%m-%d"),)
    )

    # -----------------------------------
    # Insert into History Table
    # -----------------------------------

    monitor_conn.execute("""
    INSERT INTO insertion_history
    VALUES (?, ?, ?, ?, ?)
    """, (
        today.strftime("%Y-%m-%d %H:%M:%S"),
        bitcoin_db,
        prediction_db,
        bitcoin_csv,
        forecast_csv
    ))

    monitor_conn.commit()

    print(f"Bitcoin DB Created: {bitcoin_db}")
    print(f"Prediction DB Created: {prediction_db}")
    print(f"Bitcoin CSV Created: {bitcoin_csv}")
    print(f"Forecast CSV Created: {forecast_csv}")

else:
    print("Less than 10 days. No new DB created.")

# -----------------------------------
# STEP 5: Close Monitoring DB
# -----------------------------------

monitor_conn.close()

print("\nProcess Completed Successfully!")

Bitcoin DB Created: bitcoin_20260512.db
Prediction DB Created: bitcoin_prediction_20260512.db
Bitcoin CSV Created: bitcoin_20260512.csv
Forecast CSV Created: bitcoin_forecast_20260512.csv

Process Completed Successfully!


c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\sidha\AppData\Local\Programs\Python\Python310\lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is ava